# Example-02: Estimation of parameters (harmonic sum & batched mode)

In [1]:
# Import

import numpy
import torch
import yaml

import sys
sys.path.append('..')

from harmonica.util import mod
from harmonica.window import Window
from harmonica.data import Data
from harmonica.frequency import Frequency
from harmonica.filter import Filter
from harmonica.decomposition import Decomposition

import matplotlib.pyplot as plt

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

True


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Estimate parameters and errors for a batch of frequencies

# Set parameters

length = 1024

# Set window

w = Window.from_cosine(length, 1.0, dtype=dtype, device=device)

# Set noise parameters

s_x = torch.tensor([0.05, 0.01], dtype=dtype, device=device)
s_f = 1.0E-5

# Set data

t = torch.linspace(0, length - 1, length, dtype=dtype, device=device)
x1 = torch.stack([0.25*torch.cos(2.0*numpy.pi*0.12*t + 0.10), 0.75*torch.cos(2.0*numpy.pi*0.12*t + 0.60)])
x2 = torch.stack([0.10*torch.cos(2.0*numpy.pi*0.24*t + 0.25), 0.25*torch.cos(2.0*numpy.pi*0.24*t + 0.15)])
x3 = torch.stack([0.05*torch.cos(2.0*numpy.pi*0.36*t + 0.10), 0.01*torch.cos(2.0*numpy.pi*0.36*t + 0.50)])
x = x1 + x2 + x3

# Compute parameters and errors for each frequency (direct)

p1a, s1a = Decomposition.harmonic_sum(0.12, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p1b, s1b = Decomposition.harmonic_sum(0.24, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p1c, s1c = Decomposition.harmonic_sum(0.36, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p1 = torch.stack([p1a, p1b, p1c])
s1 = torch.stack([s1a, s1b, s1c])

# Compute parameters and errors for each frequency (automatic)

p2a, s2a = Decomposition.harmonic_sum_automatic(0.12, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p2b, s2b = Decomposition.harmonic_sum_automatic(0.24, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p2c, s2c = Decomposition.harmonic_sum_automatic(0.36, w.window, x, error=True, sigma=s_x, sigma_frequency=s_f)
p2 = torch.stack([p2a, p2b, p2c])
s2 = torch.stack([s2a, s2b, s2c])

# Compute parameters and errors for each frequency (batched)

p3, s3 = Decomposition.harmonic_sum_batched(
    torch.tensor([0.12, 0.24, 0.36], dtype=dtype, device=device),
    w.window,
    x,
    error=True,
    sigma=s_x,
    sigma_frequency=torch.tensor([s_f, s_f, s_f], dtype=dtype, device=device))

# Compare errors

print(s1)
print(s2)
print(s3)

tensor([[[2.822920601761e-03, 8.447544296222e-03, 2.706329386825e-03, 3.394245973862e-02],
         [1.363412168625e-02, 1.992058457734e-02, 5.412658773653e-04, 3.217800385559e-02]],

        [[2.820933989700e-03, 4.127929982633e-03, 2.706329386827e-03, 4.203956358381e-02],
         [1.318111090385e-03, 7.970572419377e-03, 5.412658773716e-04, 3.224269320318e-02]],

        [[2.711089283224e-03, 3.144152029843e-03, 2.706329386826e-03, 6.296499040365e-02],
         [5.628105253876e-04, 6.104693260486e-04, 5.412658773675e-04, 6.296490438044e-02]]],
       dtype=torch.float64)
tensor([[[2.822920601761e-03, 8.447544296222e-03, 2.706329386825e-03, 3.394245973862e-02],
         [1.363412168625e-02, 1.992058457734e-02, 5.412658773653e-04, 3.217800385559e-02]],

        [[2.820933989700e-03, 4.127929982633e-03, 2.706329386827e-03, 4.203956358381e-02],
         [1.318111090385e-03, 7.970572419377e-03, 5.412658773716e-04, 3.224269320318e-02]],

        [[2.711089283224e-03, 3.144152029843e-03, 2.

In [4]:
# Estimate parameters for a list of harmonics

# Set parameters

length = 2048

# Set window

w = Window.from_cosine(length, 4.0, dtype=dtype, device=device)

# Set data

t = torch.linspace(0, length - 1, length, dtype=dtype, device=device)
x1 = torch.stack([0.25*torch.cos(2.0*numpy.pi*0.12*t + 0.10)])
x2 = torch.stack([0.10*torch.cos(2.0*numpy.pi*0.24*t + 0.25)])
x3 = torch.stack([0.05*torch.cos(2.0*numpy.pi*0.36*t + 0.10)])
x = x1 + x2 + x3

# Set harmonics

f = 0.12
h = torch.tensor([*Frequency.harmonics(10, [f]).values()], dtype=dtype, device=device)

# Estimate parameters

param, _ = Decomposition.harmonic_sum_batched(h, w.window, x, error=False)

# Result (amplitude)

*_, a, _ = param.swapaxes(0, -1)
print(a.T)

tensor([[2.500000000000e-01],
        [1.000000000000e-01],
        [5.000000000000e-02],
        [2.623619031558e-16],
        [4.015512494878e-16],
        [3.147267753522e-16],
        [1.854150954133e-16],
        [1.135456674832e-16],
        [7.519772959361e-16],
        [2.807503616015e-16]], dtype=torch.float64)
